# 🟡 Google Data Engineer – Page 2 (Q420-Q709)
| # | Title | Category | Difficulty |
|---|-------|----------|-----------|
| Q427 | Flagging Metrics Summary | SQL | HARD |
| Q464 | Top 10 Highest-Rated Hotels | SQL | MEDIUM |
| Q503 | Meeting Rooms | Python | EASY |
| Q546 | Share of Shippable Orders | SQL | MEDIUM |
| Q578 | Average Pay per Education | SQL | EASY |
| Q595 | Survivors by Sex and Class | SQL | MEDIUM |
| Q662 | Accounts Merge | Python | MEDIUM |


---
## Q427 · Flagging Metrics Summary (Database – HARD)

### Problem
Given `flags` (user_id, video_id, flag_reason, reviewed, is_correct) table,
compute for each flag_reason: total flags, % reviewed, % correct when reviewed.

---
### 🧠 How to Remember
```
1. Three metrics → three CASE WHEN counts in one GROUP BY
2. % reviewed = SUM(reviewed=1) / COUNT(*) * 100
3. % correct  = SUM(is_correct=1 AND reviewed=1) / SUM(reviewed=1) * 100
4. NULLIF to avoid division by zero
```


In [ ]:
import sqlite3
import pandas as pd

conn2 = sqlite3.connect(":memory:")
conn2.execute("""
CREATE TABLE flags (
    user_id INT, video_id INT, flag_reason TEXT,
    reviewed INT, is_correct INT
)
""")
conn2.executemany("INSERT INTO flags VALUES (?,?,?,?,?)", [
    (1, 101, 'spam',       1, 1),
    (2, 102, 'spam',       1, 0),
    (3, 103, 'hate_speech',1, 1),
    (4, 104, 'spam',       0, None),
    (5, 105, 'hate_speech',1, 1),
    (6, 106, 'violence',   0, None),
    (7, 107, 'spam',       1, 1),
    (8, 108, 'violence',   1, 0),
])

# Solution 1: CASE WHEN aggregation
sql_flag1 = """
SELECT
    flag_reason,
    COUNT(*) AS total_flags,
    ROUND(100.0 * SUM(reviewed) / COUNT(*), 1) AS pct_reviewed,
    ROUND(100.0 * SUM(CASE WHEN reviewed=1 AND is_correct=1 THEN 1 ELSE 0 END)
          / NULLIF(SUM(reviewed), 0), 1) AS pct_correct_when_reviewed
FROM flags
GROUP BY flag_reason
ORDER BY total_flags DESC
"""
print("Solution 1 - CASE WHEN:")
print(pd.read_sql(sql_flag1, conn2))


In [ ]:
# Solution 2: CTE for readability
sql_flag2 = """
WITH base AS (
    SELECT flag_reason,
           COUNT(*) AS total,
           SUM(reviewed) AS n_reviewed,
           SUM(CASE WHEN reviewed=1 AND is_correct=1 THEN 1 ELSE 0 END) AS n_correct
    FROM flags
    GROUP BY flag_reason
)
SELECT flag_reason, total AS total_flags,
       ROUND(100.0 * n_reviewed / total, 1) AS pct_reviewed,
       ROUND(100.0 * n_correct / NULLIF(n_reviewed, 0), 1) AS pct_correct_when_reviewed
FROM base
ORDER BY total_flags DESC
"""
print("Solution 2 - CTE approach:")
print(pd.read_sql(sql_flag2, conn2))


In [ ]:
# Solution 3: Window functions + FILTER clause style
sql_flag3 = """
SELECT DISTINCT flag_reason,
    COUNT(*) OVER (PARTITION BY flag_reason) AS total_flags,
    ROUND(100.0 * SUM(reviewed) OVER (PARTITION BY flag_reason)
          / COUNT(*) OVER (PARTITION BY flag_reason), 1) AS pct_reviewed,
    ROUND(100.0 * SUM(CASE WHEN reviewed=1 AND is_correct=1 THEN 1 ELSE 0 END)
                  OVER (PARTITION BY flag_reason)
          / NULLIF(SUM(reviewed) OVER (PARTITION BY flag_reason), 0), 1) AS pct_correct
FROM flags
ORDER BY total_flags DESC
"""
print("Solution 3 - Window functions:")
print(pd.read_sql(sql_flag3, conn2))


---
## Q464 · Top 10 Highest-Rated Hotels (Database – MEDIUM)

### Problem
Given `hotel_reviews` (hotel_name, review_score), find the top 10 hotels by average review score.

---
### 🧠 How to Remember
```
1. AVG(review_score) GROUP BY hotel_name
2. ORDER BY avg DESC
3. LIMIT 10
4. For ties: use RANK() or DENSE_RANK()
```


In [ ]:
conn2.execute("""
CREATE TABLE hotel_reviews (hotel_name TEXT, review_score REAL, review_date TEXT)
""")
import random
random.seed(42)
hotels = ['Grand Plaza','City Inn','Beach Resort','Mountain Lodge','Airport Hotel',
          'Downtown Suites','Riverside Hotel','Garden View','Sky Tower','Harbor Inn',
          'Central Park','Budget Stay']
data = [(h, round(random.uniform(3.0,5.0),1), '2023-01-01') for h in hotels for _ in range(random.randint(5,15))]
conn2.executemany("INSERT INTO hotel_reviews VALUES (?,?,?)", data)

# Solution 1: Simple GROUP BY + LIMIT
sql_hotel1 = """
SELECT hotel_name, ROUND(AVG(review_score),2) AS avg_score, COUNT(*) AS review_count
FROM hotel_reviews
GROUP BY hotel_name
ORDER BY avg_score DESC
LIMIT 10
"""
print("Solution 1 - Simple TOP 10:")
print(pd.read_sql(sql_hotel1, conn2))


In [ ]:
# Solution 2: RANK() to handle ties properly
sql_hotel2 = """
WITH hotel_stats AS (
    SELECT hotel_name,
           ROUND(AVG(review_score),2) AS avg_score,
           COUNT(*) AS review_count
    FROM hotel_reviews
    GROUP BY hotel_name
)
SELECT hotel_name, avg_score, review_count,
       RANK() OVER (ORDER BY avg_score DESC) AS rank
FROM hotel_stats
WHERE RANK() OVER (ORDER BY avg_score DESC) <= 10
"""
# SQLite doesn't support window in WHERE directly, use subquery:
sql_hotel2 = """
WITH hotel_stats AS (
    SELECT hotel_name,
           ROUND(AVG(review_score),2) AS avg_score,
           COUNT(*) AS review_count
    FROM hotel_reviews GROUP BY hotel_name
),
ranked AS (
    SELECT *, RANK() OVER (ORDER BY avg_score DESC) AS rnk FROM hotel_stats
)
SELECT hotel_name, avg_score, review_count, rnk FROM ranked WHERE rnk <= 10
"""
print("Solution 2 - With RANK():")
print(pd.read_sql(sql_hotel2, conn2))


In [ ]:
# Solution 3: Weighted by recency (advanced)
sql_hotel3 = """
WITH weighted AS (
    SELECT hotel_name,
           ROUND(AVG(review_score),2) AS avg_score,
           COUNT(*) AS review_count,
           MIN(review_date) AS first_review,
           MAX(review_date) AS last_review
    FROM hotel_reviews GROUP BY hotel_name
)
SELECT hotel_name, avg_score, review_count,
       CASE WHEN review_count >= 10 THEN 'High confidence'
            WHEN review_count >= 5  THEN 'Medium confidence'
            ELSE 'Low confidence' END AS confidence
FROM weighted
ORDER BY avg_score DESC
LIMIT 10
"""
print("Solution 3 - With confidence bands:")
print(pd.read_sql(sql_hotel3, conn2))


---
## Q503 · Meeting Rooms (Data Structures – EASY)

### Problem
Given intervals [[start,end]] of meetings, determine if a person can attend **all** meetings
(i.e., no two meetings overlap).

**Example:** [[0,30],[5,10],[15,20]] → False (overlap at 5)

---
### 🧠 How to Remember
```
1. SORT by start time
2. Check if any meeting starts before previous ends
3. If intervals[i][0] < intervals[i-1][1] → OVERLAP → return False
```


In [ ]:
# Solution 1: Brute force - check all pairs
# Time: O(n²)  Space: O(1)
def can_attend_meetings_brute(intervals: List[List[int]]) -> bool:
    """Check every pair for overlap."""
    n = len(intervals)
    for i in range(n):
        for j in range(i+1, n):
            a, b = intervals[i], intervals[j]
            # Overlap if one starts before other ends
            if not (a[1] <= b[0] or b[1] <= a[0]):
                return False
    return True

print("Solution 1:", can_attend_meetings_brute([[0,30],[5,10],[15,20]]))  # False
print("Solution 1:", can_attend_meetings_brute([[7,10],[2,4]]))  # True


In [ ]:
# Solution 2: Sort + Linear Scan ← OPTIMAL
# Time: O(n log n)  Space: O(1)
def can_attend_meetings_sort(intervals: List[List[int]]) -> bool:
    """Sort by start, check adjacent pairs only."""
    intervals.sort(key=lambda x: x[0])
    for i in range(1, len(intervals)):
        if intervals[i][0] < intervals[i-1][1]:  # new starts before prev ends
            return False
    return True

print("Solution 2:", can_attend_meetings_sort([[0,30],[5,10],[15,20]]))  # False
print("Solution 2:", can_attend_meetings_sort([[7,10],[2,4]]))  # True


In [ ]:
# Solution 3: Event timeline (sweep line)
# Time: O(n log n)  Space: O(n)
def can_attend_meetings_sweep(intervals: List[List[int]]) -> bool:
    """Flatten to events: +1 at start, -1 at end. Overlap if count > 1."""
    events = []
    for start, end in intervals:
        events.append((start, 1))   # meeting starts
        events.append((end,  -1))   # meeting ends
    events.sort()

    active = 0
    for time, delta in events:
        active += delta
        if active > 1:  # more than 1 meeting at same time
            return False
    return True

print("Solution 3:", can_attend_meetings_sweep([[0,30],[5,10],[15,20]]))  # False
print("Solution 3:", can_attend_meetings_sweep([[7,10],[2,4]]))  # True
print("\n⏱ Complexity Summary:")
print("| Solution   | Time      | Space |")
print("|------------|-----------|-------|")
print("| Brute      | O(n²)     | O(1)  |")
print("| Sort ✅    | O(n log n)| O(1)  |")
print("| Sweep Line | O(n log n)| O(n)  |")


---
## Q546 · Share of Shippable Orders (Database – MEDIUM)

### Problem
Given `orders` (order_id, item_id, quantity) and `inventory` (item_id, stock),
find % of orders where ALL items have sufficient stock.

---
### 🧠 How to Remember
```
1. JOIN orders with inventory
2. Flag each order-item pair: can_ship = (stock >= quantity)
3. Per order: all items must be shippable → MIN(can_ship) = 1
4. % shippable = COUNT(shippable orders) / COUNT(all orders) * 100
```


In [ ]:
conn2.execute("CREATE TABLE orders (order_id INT, item_id INT, quantity INT)")
conn2.execute("CREATE TABLE inventory (item_id INT, stock INT)")
conn2.executemany("INSERT INTO orders VALUES (?,?,?)", [
    (1, 10, 2), (1, 20, 3),
    (2, 10, 1), (2, 30, 5),
    (3, 20, 10),
    (4, 10, 1),
])
conn2.executemany("INSERT INTO inventory VALUES (?,?)", [
    (10, 5), (20, 2), (30, 10),
])

# Solution 1: Subquery approach
sql_ship1 = """
WITH order_check AS (
    SELECT o.order_id,
           MIN(CASE WHEN i.stock >= o.quantity THEN 1 ELSE 0 END) AS fully_shippable
    FROM orders o
    JOIN inventory i ON o.item_id = i.item_id
    GROUP BY o.order_id
)
SELECT
    COUNT(CASE WHEN fully_shippable = 1 THEN 1 END) AS shippable_orders,
    COUNT(*) AS total_orders,
    ROUND(100.0 * COUNT(CASE WHEN fully_shippable=1 THEN 1 END) / COUNT(*), 1) AS pct_shippable
FROM order_check
"""
print("Solution 1 - Shippable orders:")
print(pd.read_sql(sql_ship1, conn2))


In [ ]:
# Solution 2: NOT EXISTS approach (exclude orders with any unshippable item)
sql_ship2 = """
SELECT
    (SELECT COUNT(DISTINCT order_id) FROM orders o
     WHERE NOT EXISTS (
         SELECT 1 FROM inventory i
         WHERE i.item_id = o.item_id AND i.stock < o.quantity
     )) AS shippable,
    COUNT(DISTINCT order_id) AS total,
    ROUND(100.0 * (SELECT COUNT(DISTINCT order_id) FROM orders o2
          WHERE NOT EXISTS (SELECT 1 FROM inventory i2
                            WHERE i2.item_id=o2.item_id AND i2.stock < o2.quantity))
          / COUNT(DISTINCT order_id), 1) AS pct_shippable
FROM orders
"""
print("Solution 2 - NOT EXISTS:")
print(pd.read_sql(sql_ship2, conn2))


In [ ]:
# Solution 3: HAVING clause style
sql_ship3 = """
WITH item_check AS (
    SELECT o.order_id, o.item_id, o.quantity, i.stock,
           CASE WHEN i.stock >= o.quantity THEN 1 ELSE 0 END AS ok
    FROM orders o LEFT JOIN inventory i ON o.item_id = i.item_id
),
order_status AS (
    SELECT order_id,
           CASE WHEN MIN(ok) = 1 THEN 'Shippable' ELSE 'Blocked' END AS status
    FROM item_check
    GROUP BY order_id
)
SELECT status, COUNT(*) AS cnt,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS pct
FROM order_status GROUP BY status
"""
print("Solution 3 - Order status breakdown:")
print(pd.read_sql(sql_ship3, conn2))


---
## Q662 · Accounts Merge (Data Structures – MEDIUM)

### Problem
Given a list of accounts where account[0] is the name and rest are emails,
merge accounts that share at least one email. Return merged list sorted.

**Example:**
```
[["John","j@g.com","j@c.com"],["John","j@c.com","j@g.com"],["Mary","m@g.com"]]
→ [["John","j@c.com","j@g.com"],["Mary","m@g.com"]]
```

---
### 🧠 How to Remember
```
UNION FIND pattern for grouping:
1. Map each email → account index
2. If email seen before → UNION current account with previous
3. Group all emails by their root account
4. Sort emails within each group, prepend name
```


In [ ]:
# Solution 1: DFS Graph approach
# Time: O(n*k log(n*k))  Space: O(n*k) where k=avg emails per account
from collections import defaultdict

def accounts_merge_dfs(accounts: List[List[str]]) -> List[List[str]]:
    """Build email graph, DFS to find connected components."""
    graph = defaultdict(set)
    email_to_name = {}

    for account in accounts:
        name = account[0]
        for email in account[1:]:
            email_to_name[email] = name
            graph[account[1]].add(email)   # connect all to first email
            graph[email].add(account[1])

    visited = set()
    result = []

    def dfs(email, component):
        visited.add(email)
        component.append(email)
        for neighbor in graph[email]:
            if neighbor not in visited:
                dfs(neighbor, component)

    for email in email_to_name:
        if email not in visited:
            component = []
            dfs(email, component)
            result.append([email_to_name[email]] + sorted(component))

    return result

accounts = [["John","j@g.com","j@c.com"],["John","j@c.com","j@g.com"],["Mary","m@g.com"]]
print("Solution 1 (DFS):", accounts_merge_dfs(accounts))


In [ ]:
# Solution 2: Union Find (Disjoint Set Union)
# Time: O(n*k * α(n))  Space: O(n*k)
class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # path compression
        return self.parent[x]
    def union(self, x, y):
        self.parent[self.find(x)] = self.find(y)

def accounts_merge_uf(accounts: List[List[str]]) -> List[List[str]]:
    uf = UnionFind()
    email_to_name = {}

    for account in accounts:
        name, emails = account[0], account[1:]
        for email in emails:
            email_to_name[email] = name
            uf.union(emails[0], email)   # union all emails in account

    # Group emails by root
    groups = defaultdict(list)
    for email in email_to_name:
        groups[uf.find(email)].append(email)

    return [[email_to_name[root]] + sorted(emails) for root, emails in groups.items()]

accounts2 = [["John","j@g.com","j@c.com"],["John","j@c.com","j@g.com"],["Mary","m@g.com"]]
print("Solution 2 (Union Find):", accounts_merge_uf(accounts2))


In [ ]:
# Solution 3: BFS approach (alternative to DFS)
# Time: O(n*k log(n*k))  Space: O(n*k)
from collections import deque

def accounts_merge_bfs(accounts: List[List[str]]) -> List[List[str]]:
    """BFS traversal of email graph."""
    graph = defaultdict(set)
    email_to_name = {}

    for account in accounts:
        name = account[0]
        for email in account[1:]:
            email_to_name[email] = name
            graph[account[1]].add(email)
            graph[email].add(account[1])

    visited = set()
    result = []

    for start_email in email_to_name:
        if start_email not in visited:
            queue = deque([start_email])
            component = []
            while queue:
                email = queue.popleft()
                if email in visited: continue
                visited.add(email)
                component.append(email)
                queue.extend(graph[email] - visited)
            result.append([email_to_name[start_email]] + sorted(component))

    return result

accounts3 = [["John","j@g.com","j@c.com"],["John","j@c.com","j@g.com"],["Mary","m@g.com"]]
print("Solution 3 (BFS):", accounts_merge_bfs(accounts3))
print("\n⏱ Complexity Summary:")
print("| Solution   | Time           | Space  |")
print("|------------|----------------|--------|")
print("| DFS        | O(nk log(nk))  | O(nk)  |")
print("| Union Find | O(nk·α(n))     | O(nk)  |")
print("| BFS        | O(nk log(nk))  | O(nk)  |")
